# 🗳️ Notebook 1 — Quorum Basics: From Single Node to W + R > N

Welcome! In this lab we build a tiny **replicated key–value store** from scratch
and discover the rule that powers Amazon DynamoDB, Apache Cassandra, and Riak:

> **If `W + R > N` then every read overlaps with the latest write → strong consistency.**

We'll walk a **bad → better → best** progression:

| Step | Replication | Consistency | Availability |
|------|-------------|-------------|--------------|
| ❌ 1. Single node | none | fine (1 copy) | **dies** if node dies |
| ⚠️ 2. Replicated, `W=R=1` | N copies | **stale reads** possible | very high |
| ✅ 3. Quorum `W+R>N` | N copies | **strong** | slightly lower |

No prior distributed-systems knowledge required — if you know Python lists and
dictionaries, you're good.


## 🛠️ Setup

From a terminal, in `02-distributed-primitives/quorum`:

```bash
uv sync
```

Then in VS Code pick the `.venv` kernel (top-right of this notebook). If it
doesn't appear, `Cmd+Shift+P` → **Reload Window**.


## 🧠 The intuition (pigeonhole principle)

Imagine **N = 5** replicas drawn as 5 boxes.

- A write with **W = 3** must touch **3** of them.
- A read with **R = 3** must touch **3** of them.

Can you pick two sets of 3 boxes out of 5 that **don't share at least one box**?
No — there are only 5 boxes, so 3 + 3 = 6 pigeons into 5 holes means **at least
one box is in both sets**. That shared box is the one holding the latest value,
and since we ask every reader for a timestamp and take the newest, we're safe.

That's the whole trick: **`W + R > N` forces an overlap**.


## ❌ Step 1 — A single node (no replication)

The simplest "database" is a Python dict. Works great… until the process dies.


In [1]:
class SingleNodeStore:
    def __init__(self):
        self.data = {}
        self.alive = True

    def write(self, k, v):
        if not self.alive:
            raise RuntimeError("node is down!")
        self.data[k] = v

    def read(self, k):
        if not self.alive:
            raise RuntimeError("node is down!")
        return self.data.get(k)

store = SingleNodeStore()
store.write("user:42", "Ada")
print("read:", store.read("user:42"))

# now the node crashes
store.alive = False
try:
    store.read("user:42")
except RuntimeError as e:
    print("💥", e)


read: Ada
💥 node is down!


**Lesson:** a single node is a single point of failure. Let's replicate the
data across several machines.

## 🧱 A replicated cluster

Each replica keeps its own copy of `(value, timestamp)` per key. Writes fan out
to **every** replica; reads contact `R` of them and pick the one with the
**highest timestamp** (last-write-wins conflict resolution).


In [2]:
from dataclasses import dataclass, field
from typing import Dict, Tuple
import random

@dataclass
class Replica:
    name: str
    up: bool = True
    data: Dict[str, Tuple[str, int]] = field(default_factory=dict)

    def write(self, k, v, ts):
        if not self.up:
            return False                       # can't reach a dead replica
        cur = self.data.get(k)
        if cur is None or ts > cur[1]:         # keep freshest by timestamp
            self.data[k] = (v, ts)
        return True

    def read(self, k):
        if not self.up:
            return None
        return self.data.get(k)                # (value, ts) or None


class Cluster:
    def __init__(self, n=5):
        self.replicas = [Replica(f"r{i}") for i in range(n)]
        self.n = n
        self._ts = 0                           # central clock for simplicity

    def _next_ts(self):
        self._ts += 1
        return self._ts

    def write(self, key, value, W):
        '''Send write to every replica. Succeed if >= W acknowledge.'''
        ts = self._next_ts()
        acks = sum(1 for r in self.replicas if r.write(key, value, ts))
        return (acks >= W), ts, acks

    def read(self, key, R):
        '''Contact R live replicas, return the freshest response.'''
        responses = []
        for r in self.replicas:
            if not r.up:
                continue
            responses.append((r.name, r.read(key)))
            if len(responses) >= R:
                break
        if len(responses) < R:
            return None                        # not enough replicas → fail
        with_data = [x for x in responses if x[1] is not None]
        if not with_data:
            return None
        return max(with_data, key=lambda x: x[1][1])   # freshest ts wins


## ⚠️ Step 2 — Replicated but W = R = 1 (eventual consistency)

High availability: any single live replica can answer. But a slow/partitioned
replica can return **stale** data.

**Scenario:**
1. Write `v1` to all 5 replicas (everyone is in sync).
2. A network partition isolates `r1..r4`. Only `r0` is reachable.
3. Client writes `v2` with `W=1` — only `r0` gets it.
4. Partition heals. Client reads with `R=1` from a random replica.

Roughly **4 out of 5** reads will return the **stale** `v1`.


In [3]:
random.seed(42)
c = Cluster(n=5)
c.write("x", "v1", W=5)                        # everyone has v1

# partition: r1..r4 unreachable
for r in c.replicas[1:]:
    r.up = False
c.write("x", "v2", W=1)                        # only r0 is updated

# heal
for r in c.replicas:
    r.up = True

stale = fresh = 0
for _ in range(1000):
    random.shuffle(c.replicas)                 # randomise who answers first
    value = c.read("x", R=1)[1][0]
    if value == "v1":
        stale += 1
    else:
        fresh += 1
print(f"R=1 reads over 1000 tries → fresh={fresh}, stale={stale}")
print("That's ~80% stale reads — clearly not OK for e.g. a bank balance.")


R=1 reads over 1000 tries → fresh=202, stale=798
That's ~80% stale reads — clearly not OK for e.g. a bank balance.


## ✅ Step 3 — Quorum: W + R > N (strong consistency)

Same partition scenario, but now we write with `W=3` and read with `R=3`.
`3 + 3 = 6 > 5 = N`, so every read must **overlap with the write set** and
therefore sees `v2`.


In [4]:
random.seed(7)
c = Cluster(n=5)
c.write("x", "v1", W=5)

# partition: r1 and r2 are unreachable
c.replicas[1].up = False
c.replicas[2].up = False
ok, ts, acks = c.write("x", "v2", W=3)
print(f"write v2 with W=3 → ok={ok}, acks={acks}  (landed on r0, r3, r4)")

# heal
for r in c.replicas:
    r.up = True

for _ in range(5):
    random.shuffle(c.replicas)
    print("R=3 read →", c.read("x", R=3))


write v2 with W=3 → ok=True, acks=3  (landed on r0, r3, r4)
R=3 read → ('r4', ('v2', 2))
R=3 read → ('r3', ('v2', 2))
R=3 read → ('r0', ('v2', 2))
R=3 read → ('r0', ('v2', 2))
R=3 read → ('r3', ('v2', 2))


Every read returns `v2`. That's the W + R > N guarantee in action.

## 📊 Cheat sheet (N = 5)

| W | R | W+R | Guarantee        | Good for                                  |
|---|---|-----|------------------|-------------------------------------------|
| 5 | 1 |  6  | strong           | read-heavy; can tolerate write outages    |
| 1 | 5 |  6  | strong           | write-heavy logs; reads can be slow       |
| 3 | 3 |  6  | strong, balanced | typical OLTP                              |
| 2 | 2 |  4  | **eventual**     | fast everything, staleness OK             |
| 1 | 1 |  2  | **eventual**     | max availability (DNS, shopping carts)    |

Picking `W` and `R` is the main **dial** in Dynamo-style databases between
consistency, availability, and latency.


## 🎯 Try it yourself

1. Set `N=7`, `W=4`, `R=4`. Partition any 3 replicas. Do reads still see the
   latest write?
2. Set `W=R=2` with `N=5`. Can you construct a partition where a read misses
   the write? (Hint: write lands on `{r0, r1}`, read goes to `{r2, r3}`.)
3. What happens to write availability as `W` grows toward `N`? We'll quantify
   this in **Notebook 2**.
